<div dir="rtl">

# Jev + Claude: סיווג, חילוץ ואימות של מסמכים רפואיים

התהליך במחברת הזו משלב שני סוגי מודלים, כל אחד במה שהוא טוב בו:

| שלב | מודל | מה הוא עושה | למה הוא |
|---|---|---|---|
| 1. סיווג | **Jev** (System One) | סוג המסמך, תחום רפואי, דחיפות, האם יש פרטים מזהים | החלטה סגורה, מהירה וזולה, עם ביטחון מכויל |
| 2. חילוץ | **Claude** (`claude-opus-5-5`) | שדות מובנים: אבחנה, קוד ICD-10, תרופות, ערכי מעבדה, המשך טיפול | צריך לקרוא ולכתוב ערכים, ו-Jev לא כותב טקסט |
| 3. אימות | **Jev** | לכל שדה שחולץ: האם המסמך באמת תומך בו? ועוד ציון איכות כולל | שאלה סגורה של כן/לא, עם הסתברות |
| ניתוב | קוד | קבלה אוטומטית, בדיקה מהירה או בדיקה אנושית, לפי הספים | |

**שני מצבי הרצה**
- `live`: קריאות אמיתיות ל-Jev ול-Claude. צריך `TYPESAFE_API_KEY` ו-`ANTHROPIC_API_KEY`.
- `demo`: רץ בלי מפתחות ובלי רשת. **ה-Jev בדמו הוא היוריסטיקה מילולית ולא המודל**, וה"חילוץ" משחזר את התשובה הנכונה מקובץ הייחוס. כל המספרים בדמו ממחישים את הזרימה בלבד ואינם תוצאות של מודל.

> ⚠️ כל המסמכים בתיקייה `documents/` סינתטיים ובדויים. זה לא כלי קליני ואין להשתמש בו לקבלת החלטות רפואיות.

</div>

In [1]:
# בהרצה ב-Colab או בסביבה חדשה:
# %pip install -q typesafe-sdk anthropic pydantic pandas
import os, json
import pandas as pd
import pipeline as P

pd.set_option("display.max_colwidth", 80)
MODE = None            # None = live אם שני המפתחות מוגדרים, אחרת demo. אפשר לכפות "live" או "demo"
INJECT_ERRORS = True   # מבחן כאוס: שותל 2 טעויות חילוץ כדי לראות ששלב 3 תופס אותן

mode, jev, extractor = P.make_backends(MODE)
print(f"mode: {mode}\nstage 1+3: {jev.name}\nstage 2:   {extractor.name}")

mode: demo
stage 1+3: DemoJev (heuristic stand-in)
stage 2:   DemoExtractor (reference replay)


<div dir="rtl">

## המסמכים

11 מסמכים סינתטיים מ-10 סוגים, כל אחד עם אבחנה אחרת. שניים מהם קשים במיוחד:
- **`09_ct_head_negative`**: פענוח CT מלא בשלילות ("ללא עדות לשבר", "ללא דימום"). מודל שמחלץ בחיפזון עלול לרשום שבר שלא קיים.
- **`11_mixed_fax`**: פקס סרוק עם שגיאות OCR, שמשלב הפניה ותוצאות מעבדה.

</div>

In [2]:
docs = P.load_documents()
gt = P.load_ground_truth()
pd.DataFrame([{"doc_id": d.doc_id, "סוג (ייחוס)": gt[d.doc_id]["doc_type"],
               "אבחנה (ייחוס)": gt[d.doc_id]["extraction"]["primary_diagnosis"],
               "ICD-10": gt[d.doc_id]["extraction"]["icd10_code"], "מילים": len(d.text.split())} for d in docs])

,doc_id,סוג (ייחוס),אבחנה (ייחוס),ICD-10,מילים
0,01_discharge_pneumonia,discharge_summary,"דלקת ריאות נרכשת בקהילה, אונה תחתונה ימנית",J18.1,176
1,02_lab_diabetes,lab_report,"סוכרת סוג 2 שאינה מאוזנת, עם מיקרואלבומינוריה",E11.65,108
2,03_er_stemi,er_visit,אוטם חריף בשריר הלב עם הרמת ST (STEMI) בדופן הקדמית,I21.0,120
3,04_imaging_fracture,imaging_report,שבר Colles ברדיוס הדיסטלי השמאלי,S52.532A,95
4,05_pathology_breast,pathology_report,"קרצינומה דוקטלית חודרנית, דרגה היסטולוגית 2",C50.412,98
5,06_referral_hypothyroid,referral_letter,תת-פעילות של בלוטת התריס על רקע דלקת אוטואימונית (השימוטו),E06.3,101
6,07_prescription_hypertension,prescription,יתר לחץ דם ראשוני,I10,85
7,08_pediatric_otitis,clinic_visit,"דלקת אוזן תיכונה חריפה, אוזן ימין",H66.91,105
8,09_ct_head_negative,imaging_report,"חבלת ראש קלה עם נפיחות רקמה רכה בקרקפת, ללא ממצא חריף תוך-גולגולתי",S00.03XA,104
9,10_admin_physio_invoice,administrative,NaN,NaN,61


In [3]:
print(next(d for d in docs if d.doc_id == "09_ct_head_negative").text)

*** מסמך סינתטי להדגמה בלבד. כל הפרטים בדויים ***

המרכז הרפואי "שקד" (בדוי) | מכון הדימות
פענוח CT ראש ללא חומר ניגוד

שם הנבדקת: שרה אוחיון   גיל: 81   ת.ז.: 000000009 (סינתטי)
תאריך: 20/09/2026
סיבת הבדיקה: נפילה בבית עם חבלת ראש, ללא איבוד הכרה. מטופלת בנוגדי קרישה.

ממצאים:
ללא עדות לשבר בעצמות הגולגולת.
ללא עדות לדימום תוך-גולגולתי, ללא המטומה תת-דוראלית או אפי-דוראלית.
אין הזזה של קו האמצע.
נפיחות רקמה רכה בקרקפת באזור הפריאטלי הימני.
שינויים ניווניים מתאימים לגיל.

רושם:
חבלת ראש קלה עם נפיחות רקמה רכה בקרקפת. אין ממצא חריף תוך-גולגולתי.
ICD-10: S00.03XA

המלצה: השגחה קלינית לאור הטיפול בנוגדי קרישה.
ד"ר יעל רוזן (בדויה), רדיולוגית



<div dir="rtl">

## מעבר מלא על מסמך אחד: `09_ct_head_negative`

### שלב 1: Jev מסווג
ארבע שאלות נשלחות בקריאה אחת, ו-Jev עונה על כולן במקביל:

</div>

In [4]:
for name, q in P.classification_questions().items():
    crit = q.get("criteria")
    shown = list(crit) if isinstance(crit, dict) else crit
    print(f"{name:13} [{q['type']}] {q['instructions']}\n{'':16}{shown}\n")

doc_type      [choice] What kind of medical document is this?
                ['discharge_summary', 'lab_report', 'er_visit', 'imaging_report', 'pathology_report', 'referral_letter', 'prescription', 'clinic_visit', 'administrative']

specialty     [choice] Which clinical specialty should own this document?
                ['internal_medicine', 'family_medicine', 'cardiology', 'endocrinology', 'orthopedics', 'oncology', 'pediatrics', 'emergency_medicine', 'neurology', 'none']

urgency       [score] How urgently does this document require clinical action?
                ['No clinical action needed', 'Routine follow-up within weeks', 'Needs attention within days', 'Needs attention within 24-48 hours (e.g. new cancer diagnosis needing referral)', 'Life-threatening emergency, immediate action']

contains_phi  [noul] Does the document contain personally identifying details (name, ID number)?
                ['true', 'false']



In [5]:
doc = next(d for d in docs if d.doc_id == "09_ct_head_negative")
q1 = P.classification_questions()
a1, t1, _ = jev.ask(doc.text, q1 if mode == "demo" else P.wire(q1))
for k, a in a1.items():
    if a.type == "choice":
        print(f"{k:13} {a.choice:20} confidence={a.confidence:.2f}")
    elif a.type == "score":
        print(f"{k:13} score={a.score:.2f}/4         confidence={a.confidence:.2f}")
    else:
        print(f"{k:13} p(yes)={a.noul:.2f}")
print(f"latency: {t1*1000:.0f} ms")

doc_type      imaging_report       confidence=1.00
specialty     emergency_medicine   confidence=0.89
urgency       score=2.00/4         confidence=0.80
contains_phi  p(yes)=0.97
latency: 0 ms


<div dir="rtl">

### שלב 2: Claude מחלץ שדות לפי סכמה
Claude מקבל את המסמך ואת הסוג שזוהה בשלב 1, ומחזיר אובייקט `MedicalExtraction` (Pydantic) דרך structured outputs. כך הפורמט מובטח. כשמבחן הכאוס פעיל, נשתלת כאן בכוונה טעות: "שבר בגולגולת" במקום "ללא עדות לשבר".

</div>

In [6]:
ex, t2, usage = extractor.extract(doc, a1["doc_type"].choice)
if INJECT_ERRORS:
    ex, injected = P.inject_error(doc.doc_id, ex)
    print("injected (deliberate) errors in:", injected)
print(json.dumps(ex.model_dump(exclude={"evidence"}), ensure_ascii=False, indent=1))

injected (deliberate) errors in: ['primary_diagnosis', 'icd10_code']
{
 "patient_name": "שרה אוחיון",
 "patient_age": 81,
 "patient_sex": "female",
 "document_date": "2026-09-20",
 "facility": "המרכז הרפואי \"שקד\"",
 "primary_diagnosis": "שבר בעצמות הגולגולת",
 "icd10_code": "S02.0XXA",
 "secondary_diagnoses": [],
 "medications": [],
 "lab_values": [],
 "follow_up": "השגחה קלינית לאור הטיפול בנוגדי קרישה"
}


<div dir="rtl">

### שלב 3: Jev מאמת כל שדה
לכל ערך שחולץ נשאלת שאלת כן/לא: "האם המסמך אומר את זה?". ההסתברות שחוזרת היא ה-confidence של השדה. בנוסף נבדק אם קוד ה-ICD מתאים לאבחנה, ומתקבל ציון איכות כולל לחילוץ (Score, ‏0–4).

</div>

In [7]:
q3 = P.verification_questions(ex)
for k, q in q3.items():
    print(f"{k:24} [{q['type']}] {q['instructions'][:110]}")

claim_patient_age        [noul] The extractor says patient_age = «81». Is this value stated in, or directly entailed by, the document?
claim_primary_diagnosis  [noul] The extractor says primary_diagnosis = «שבר בעצמות הגולגולת». Is this value stated in, or directly entailed by
claim_icd10_code         [noul] The extractor says icd10_code = «S02.0XXA». Is this value stated in, or directly entailed by, the document?
claim_follow_up          [noul] The extractor says follow_up = «השגחה קלינית לאור הטיפול בנוגדי קרישה». Is this value stated in, or directly e
icd_consistent           [noul] Is ICD-10 code S02.0XXA a correct code for the main diagnosis described in the document?
extraction_quality       [score] Rate how complete and correct the extracted record (state.extracted) is against the document (state.document).


In [8]:
state = {"document": doc.text, "extracted": ex.model_dump(exclude={"evidence"})}
a3, t3, _ = jev.ask(state, q3 if mode == "demo" else P.wire(q3))
rows = [{"field": label, "value": value, "confidence": round(a3[k].noul, 3), "status": P.field_status(a3[k].noul)}
        for k, label, value in P.claims_from(ex)]
print(f"icd_consistent p={a3['icd_consistent'].noul:.2f}   extraction_quality={a3['extraction_quality'].score:.1f}/4")
pd.DataFrame(rows)

icd_consistent p=0.15   extraction_quality=2.0/4


,field,value,confidence,status
0,patient_age,81,0.96,accept
1,primary_diagnosis,שבר בעצמות הגולגולת,0.06,reject
2,icd10_code,S02.0XXA,0.08,reject
3,follow_up,השגחה קלינית לאור הטיפול בנוגדי קרישה,0.96,accept


In [9]:
route, reason = P.route(a1["doc_type"], {r["field"]: r["confidence"] for r in rows})
print(f"route: {route}  ({reason})")

route: human_review  (rejected fields: primary_diagnosis, icd10_code)


<div dir="rtl">

## הרצה על כל המסמכים

ספי הניתוב (ב-`pipeline.py`):
- **סיווג:** ביטחון מתחת ל-0.75 עובר למיון ידני (`human_triage`).
- **שדה:** מעל 0.90 מתקבל אוטומטית; בין 0.60 ל-0.90 עובר לבדיקה מהירה; מתחת ל-0.60 נדחה.
- **מסמך מנהלי:** עובר לארכיון בלי חילוץ.

</div>

In [10]:
results = P.run_all(docs, jev, extractor, inject=INJECT_ERRORS)
pd.DataFrame([{
    "doc_id": r["doc_id"],
    "type": r["classification"]["doc_type"]["choice"],
    "type_conf": round(r["classification"]["doc_type"]["confidence"], 2),
    "specialty": r["classification"]["specialty"]["choice"],
    "urgency": round(r["classification"]["urgency"]["score"], 1),
    "min_field_conf": min((f["confidence"] for f in r.get("fields", [])), default=None),
    "route": r["route"], "reason": r["route_reason"]} for r in results])

,doc_id,type,type_conf,specialty,urgency,min_field_conf,route,reason
0,01_discharge_pneumonia,discharge_summary,1.00,internal_medicine,1.2,0.910,auto_accept,all fields >= 0.9
1,02_lab_diabetes,lab_report,0.99,endocrinology,2.0,0.910,auto_accept,all fields >= 0.9
2,03_er_stemi,er_visit,1.00,cardiology,3.5,0.960,auto_accept,all fields >= 0.9
3,04_imaging_fracture,imaging_report,1.00,orthopedics,2.0,0.960,auto_accept,all fields >= 0.9
4,05_pathology_breast,pathology_report,1.00,oncology,2.8,0.910,auto_accept,all fields >= 0.9
5,06_referral_hypothyroid,referral_letter,0.99,endocrinology,1.2,0.960,auto_accept,all fields >= 0.9
6,07_prescription_hypertension,prescription,1.00,family_medicine,1.2,0.960,auto_accept,all fields >= 0.9
7,08_pediatric_otitis,clinic_visit,0.99,pediatrics,2.0,0.347,human_review,rejected fields: medications[0]
8,09_ct_head_negative,imaging_report,1.00,emergency_medicine,2.0,0.060,human_review,"rejected fields: primary_diagnosis, icd10_code"
9,10_admin_physio_invoice,administrative,1.00,none,0.5,NaN,archive,"non-clinical document, no extraction"


<div dir="rtl">

## הערכה מול הייחוס
- **שדה שגוי:** שדה שהערך שלו שונה מהערך בקובץ הייחוס. במצב live זה מודד גם טעויות אמיתיות של Claude, לא רק את השתולות.
- **השאלה החשובה:** האם הביטחון של שלב 3 מפריד בין שדות שגויים לשדות נכונים?

</div>

In [11]:
ev = P.evaluate(results, gt)
wrong = ev.pop("wrong_fields")
display(pd.Series(ev).to_frame("value"))
pd.DataFrame(wrong, columns=["doc_id", "field", "value", "confidence"])

,value
documents,11.00
doc_type_accuracy,1.00
specialty_accuracy,1.00
urgency_mean_abs_error,0.45
wrong_fields_flagged,3.00
wrong_fields_missed,0.00
correct_fields_flagged,0.00
correct_fields_accepted,68.00


,doc_id,field,value,confidence
0,08_pediatric_otitis,medications[0],"Amoxicillin 1350 מ""ג",0.347
1,09_ct_head_negative,primary_diagnosis,שבר בעצמות הגולגולת,0.060
2,09_ct_head_negative,icd10_code,S02.0XXA,0.080


<div dir="rtl">

## עלות וזמן
המחירים לפי מחירוני הספקים: Claude Opus 5.5 עולה ‎$4 למיליון טוקנים של קלט ו-‎$20 למיליון של פלט; Jev עולה ‎$0.042 למיליון טוקנים של קלט, והפלט חינם. במצב demo אין קריאות אמיתיות, ולכן אין טוקנים ואין זמנים אמיתיים.

</div>

In [12]:
lat = pd.DataFrame([r["latency_s"] for r in results]).describe().loc[["mean", "50%", "max"]]
display((lat * 1000).round(0).rename(index={"50%": "p50"}).add_suffix(" (ms)"))
P.cost_usd(results)

,classify (ms),extract (ms),verify (ms)
mean,0.0,0.0,1.0
p50,0.0,0.0,1.0
max,0.0,0.0,1.0


{'claude_input_tokens': 0,
 'claude_output_tokens': 0,
 'jev_input_tokens': 0,
 'claude_usd': 0.0,
 'jev_usd': 0.0}

In [13]:
os.makedirs("outputs", exist_ok=True)
out = f"outputs/results_{mode}.json"
json.dump({"mode": mode, "inject_errors": INJECT_ERRORS, "results": results, "evaluation": {**ev, "wrong_fields": wrong},
           "cost": P.cost_usd(results)}, open(out, "w"), ensure_ascii=False, indent=1, default=str)
print("saved", out)

saved outputs/results_demo.json


<div dir="rtl">

## הערות ומגבלות
- **עברית:** לא מצאתי תיעוד שמאשר שהביצועים של Jev בעברית זהים לאנגלית. לפני שימוש אמיתי צריך להריץ במצב live ולבדוק את ההערכה למעלה.
- **הספים:** 0.90, 0.60 ו-0.75 הם נקודת התחלה. כדאי לכייל אותם על נתונים אמיתיים, לפי מחיר הטעות בכל שדה. למשל, מינון תרופה מצדיק סף מחמיר יותר מתאריך.
- **שאלות האימות:** שלב 3 בודק *עקביות עם המסמך*, לא נכונות קלינית. אם המסמך עצמו שגוי, האימות לא יתפוס את זה.
- **פרטיות:** מסמכים אמיתיים מכילים מידע רפואי מזהה. צריך לוודא שהשימוש ב-API תואם את מדיניות הפרטיות והרגולציה של הארגון.

</div>